# 03. Feature Store - Feast

A **feature store** centralizes definitions, lineage, and serving (online + offline) so training and inference use consistent features. **Feast** provides a declarative layer (`feature_store.yaml` + Python definitions) on top of your data warehouse or files.


In [ ]:
from pathlib import Path
from IPython.display import display
import feast
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Feast repo path — point to aide2/feast in this project when running locally
FEAST_REPO = Path("../../feast").resolve()


## Feature Store Configuration


In [ ]:
# Typical feature_store.yaml (see aide2/feast/feature_store.yaml in repo):
yaml_text = '''
project: nyc_taxi_features
registry: s3://feast-registry/registry.db
provider: local
online_store:
  type: sqlite
  path: data/online_store.db
offline_store:
  type: file
entity_key_serialization_version: 3
auth:
  type: no_auth
'''
print(yaml_text)
# Field notes:
# - project: Feast project name for all entities/views.
# - registry: where the registry DB lives (S3/MinIO in this stack).
# - provider: orchestration backend (local dev; gcp/aws in prod).
# - online_store: low-latency serving (SQLite dev; Redis prod).
# - offline_store: batch/historical retrieval (file/Trino/Spark).
# - entity_key_serialization_version: binary key encoding for online store.
# - auth: disabled for local demos.


In [ ]:
# After placing feature definitions in features.py:
store = feast.FeatureStore(repo_path=str(FEAST_REPO))
# store.apply([...])  # register entities and views — run from CLI typically:
# feast -c <repo> apply


## Feature Definitions


In [ ]:
# Entities (from aide2/feast/features.py):
# - taxi_zone: join_keys=[location_id]
# - vendor: join_keys=[vendor_id]
# - time_bucket: join_keys=[hour_of_day]
print("Entities: taxi_zone, vendor, time_bucket")


In [ ]:
# Feature views:
# - hourly_trip_features: time_bucket, hourly_stats from gold
# - zone_features: taxi_zone, zone aggregates
# - daily_trip_features: time_bucket / daily rollups with peak_hour
print("Feature views: hourly_trip_features, zone_features, daily_trip_features")


In [ ]:
# On-demand feature view: fare_prediction_features combines zone_features + request fare
# to produce expected_fare_ratio = fare_amount / zone_avg_fare
print("On-demand: fare_prediction_features (mode='pandas')")


## Feature Materialization


In [ ]:
# Populate online store from offline sources for a time range:
# !feast materialize 2024-01-01T00:00:00 2024-01-02T00:00:00
# or in Python:
# store.materialize(start_date=..., end_date=...)
print("Run `feast materialize` from the Feast repo directory after configuring sources.")


## Online Feature Retrieval


In [ ]:
# Example entity keys: location_id=161, hour_of_day=14
entity_rows = [{"location_id": 161, "hour_of_day": 14}]
# fv = store.get_online_features(features=[...], entity_rows=entity_rows)
# Placeholder when registry/backends not available:
pd.DataFrame(
    [
        {
            "location_id": 161,
            "hour_of_day": 14,
            "avg_fare": 18.2,
            "trip_count": 120,
        }
    ]
)


In [ ]:
feat_df = pd.DataFrame(
    [
        {
            "location_id": 161,
            "hour_of_day": 14,
            "avg_fare": 18.2,
            "trip_count": 120,
            "zone_avg_fare": 17.5,
        }
    ]
)
display(feat_df)


## Historical Feature Retrieval


In [ ]:
entity_df = pd.DataFrame(
    {
        "location_id": [161, 161, 237],
        "event_timestamp": pd.to_datetime(
            ["2024-06-01 14:00:00", "2024-06-01 15:00:00", "2024-06-01 14:30:00"]
        ),
    }
)
entity_df


In [ ]:
# store.get_historical_features(entity_df=entity_df, features=[...])
hist = entity_df.copy()
hist["avg_fare_lag"] = [18.0, 17.8, 22.1]
hist["zone_trip_count"] = [400, 410, 300]
hist


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].hist(hist["avg_fare_lag"], bins=5, color="steelblue", alpha=0.8)
ax[0].set_title("avg_fare_lag distribution (sample)")
ax[1].hist(hist["zone_trip_count"], bins=5, color="coral", alpha=0.8)
ax[1].set_title("zone_trip_count distribution (sample)")
plt.tight_layout()
plt.show()


## Integration with ML Models (AIDE 1)


In [ ]:
# Training / batch inference: use get_historical_features with entity_df + timestamps
# for point-in-time correct labels. Online inference: get_online_features + on-demand views
# for real-time ratios (e.g. fare vs zone average) feeding the AIDE 1 model.

feature_vector_example = {
    "hour_of_day": 14,
    "zone_avg_fare": 17.5,
    "expected_fare_ratio": 1.04,
}
print("Model input dict:", feature_vector_example)


## Summary

Feast ties **gold-layer tables** to **entities** and **feature views**, supports **materialization** to an online store, and exposes a uniform API for **training** (historical) and **serving** (online + on-demand). Align this notebook with `aide2/feast/features.py` and your deployed registry.
